## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-20-23-53-39 +0000,startribune,Live: Mankato West holds off Monticello 34-27 ...,https://www.startribune.com/live-scores-update...
1,2026-11-20-23-52-38 +0000,nypost,Paramedic responding to deadly scene discovers...,https://nypost.com/2026/11/20/us-news/paramedi...
2,2026-11-20-23-38-43 +0000,startribune,Polaris settles with state over alleged consum...,https://www.startribune.com/polaris-settles-wi...
3,2026-11-20-23-36-00 +0000,wsj,Anthropic’s IPO Turmoil Caps $4 Trillion AI Wi...,https://www.wsj.com/finance/stocks/anthropics-...
4,2026-11-20-23-33-06 +0000,nyt,"Bobby Ray Inman, Intelligence Official Under C...",https://www.nytimes.com/2026/11/20/us/bobby-ra...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2494/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
64,trump,46
81,new,23
30,anthropic,21
35,ai,18
170,tech,17


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
170,2026-11-20-18-06-47 +0000,chitri,Donald Trump says he plans to penalize US tech...,https://www.chicagotribune.com/2026/11/20/trum...,141
89,2026-11-20-21-35-47 +0000,cbc,Trump threatens to penalize U.S. tech companie...,https://www.cbc.ca/news/world/ai-data-centres-...,125
249,2026-11-20-14-28-00 +0000,wsj,The Trump administration auctioned off parts o...,https://www.wsj.com/politics/the-u-s-is-auctio...,105
7,2026-11-20-23-29-00 +0000,wsj,During President Trump’s White House briefing ...,https://www.wsj.com/politics/elections/how-a-w...,104
73,2026-11-20-21-58-36 +0000,latimes,Trump says he'll penalize U.S. companies that ...,https://www.latimes.com/politics/story/2026-11...,104


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
170,141,2026-11-20-18-06-47 +0000,chitri,Donald Trump says he plans to penalize US tech...,https://www.chicagotribune.com/2026/11/20/trum...
86,71,2026-11-20-21-43-22 +0000,bbc,What the new UK-EU deal could mean for future ...,https://www.bbc.co.uk/news/videos/c90k6gl3lyxe...
360,67,2026-11-20-04-12-00 +0000,wsj,The Senate passed a bipartisan bill designed t...,https://www.wsj.com/politics/policy/senate-pas...
382,60,2026-11-20-01-11-52 +0000,latimes,California State Bar opens investigation into ...,https://www.latimes.com/california/story/2026-...
146,60,2026-11-20-19-39-00 +0000,wsj,"Sen. Tom Cotton (R., Ark.), a China hawk, tell...",https://www.wsj.com/politics/national-security...
394,52,2026-11-20-00-00-00 +0000,wsj,Anthropic’s Mega IPO Lives Up to Wall Street H...,https://www.wsj.com/finance/stocks/anthropic-i...
31,48,2026-11-20-22-50-36 +0000,nypost,Russia open to ‘NATO-like’ status for Ukraine ...,https://nypost.com/2026/11/20/us-news/russia-o...
114,46,2026-11-20-20-41-00 +0000,wsj,Take-Two’s ‘Grand Theft Auto VI’ Hits Nearly $...,https://www.wsj.com/business/media/take-twos-g...
281,45,2026-11-20-12-00-00 +0000,wsj,Vice President JD Vance has whipsawed between ...,https://www.wsj.com/politics/trump-says-vance-...
329,38,2026-11-20-09-18-00 +0000,wsj,"British business bank OakNorth, whose sharehol...",https://www.wsj.com/economy/central-banking/oa...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
